In [29]:
import os
from functools import partial
from timeit import default_timer as timer
from datetime import timedelta
import ast
import math
import copy


import rasterio
from rasterio.mask import mask
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape, Polygon
import pyproj
from shapely.ops import transform
import numpy as np
import pandas as pd
import geopandas as gpd
from rasterio.plot import show
import tqdm
from shapely.geometry import box
from rasterio.errors import RasterioIOError
from rasterio.enums import Resampling


In [30]:
import sys
sys.path.append('../')
from utils import get_tiles_as_gdf, extract_sample, get_window

In [31]:
target_years = list(range(2000, 2020))

In [57]:
forestloss_dir = '../../Vancutsem_yearloss'
forestloss_tiles = get_tiles_as_gdf(forestloss_dir)

In [77]:
#dataset = 'rubber'
#dataset = 'cocoa'
dataset = 'mapbiomas'


In [59]:
if dataset == 'rubber':
    rubber_data_path = '/home/jan/Documents/EPFL/project_2/data/input_datasets/rubber/Rubber_10m'
    label_gdf = get_tiles_as_gdf(rubber_data_path)
    dataset_extent = box(*label_gdf.total_bounds)
    class_mapping = {
        1 : 'rubber'
    }
    dataset_prefix = 2000000

    

elif dataset == 'mapbiomas':
    mapbiomas_path = '/home/jan/Documents/EPFL/drivers/data/mapbiomas/brasil_coverage_2022.tif'
    with rasterio.open(mapbiomas_path) as src:
        dataset_extent = box(*src.bounds)
    label_gdf = gpd.GeoDataFrame({'geometry' : [dataset_extent], 'filename': [mapbiomas_path]})
    
    class_mapping = {
        #1 : 'forest', # superclass
        #3 : 'forest',  # "Forest Formation"
        #10 : 'non-forest natural formation', #incl. wetland, grassland etc. - I think this is the 'natural regrowth'

        # Commodities & drivers
        9 : 'forest plantation',
        15 : 'pasture', 
        39 : 'soy',
        46 : 'coffee',
        35 : 'palm oil',
        30 : 'mining',
        24 : 'urban',
    }
    
    dataset_prefix = 1000000
    

elif dataset == 'cocoa':
    cocoa_file_path = '/home/jan/Documents/EPFL/project_2/data/input_datasets/cocoa_map_th_065.tif'
    with rasterio.open(cocoa_file_path) as src:
        dataset_extent = box(*src.bounds)
    label_gdf = gpd.GeoDataFrame({'geometry' : [dataset_extent], 'filename': [cocoa_file_path]})
    
    class_mapping = {
        1 : 'cocoa'
    }
    dataset_prefix = 3000000


target_values = [key for key in class_mapping.keys()]


In [60]:
filtered_forestloss = forestloss_tiles.loc[forestloss_tiles.intersects(dataset_extent)]

In [61]:
def write_numpy_array_to_tif(array, geometry, profile, patch_size, output_path):
    transform = rasterio.transform.from_bounds(
        geometry.bounds[0], 
        geometry.bounds[1], 
        geometry.bounds[2], 
        geometry.bounds[3], 
        patch_size[0], 
        patch_size[1])
    
    out_profile = copy.deepcopy(profile)

    out_profile.update({'transform': transform,
                    'width' : patch_size[0],
                    'height' : patch_size[1]
                   })
    
    with rasterio.open(output_path, "w", **out_profile) as sink:
        sink.write(array, 1)

    

In [62]:
forestloss_patches_dir = '../data/forestloss_patches/dataset_small'

In [63]:
patch_rows, patch_cols = 35, 35
min_px_forest_loss = 20 #Pixel size is 30x30m;
min_ratio_for_driver = 0.5 # E.g., for patch with 100 pixels of forest loss, at least min_ratio_for_driver*100 must 


geoms = []
forest_loss_years = []
drivers = []
sampleids = [] 
    
    
sampleid = dataset_prefix


for idx1, label_tile in label_gdf.iterrows(): #Can be a single one (cocoa, mapb) or multiple (rubber)
    print(idx1)
    forestloss_intersection = filtered_forestloss.loc[filtered_forestloss.intersects(label_tile.geometry)]
    
    for idx2, forest_loss_tile in forestloss_intersection.iterrows():
        print(idx2)
        intersection = label_tile.geometry.intersection(forest_loss_tile.geometry)
        label_window = get_window(intersection, label_tile.filename)
        forest_loss_window = get_window(intersection, forest_loss_tile.filename)

        
        with rasterio.open(forest_loss_tile.filename) as src:
            forestloss_data = src.read(1, window=forest_loss_window)
            forestloss_profile = src.profile
        
        with rasterio.open(label_tile.filename) as src:
            label_data = src.read(1,           
                out_shape=(
                    forestloss_data.shape[0],
                    forestloss_data.shape[1]  
                ),
            resampling=Resampling.bilinear,
                window=label_window)

            
        assert forestloss_data.shape == label_data.shape
        
        for i in range(0, forestloss_data.shape[0], patch_rows):
            for j in range(0, forestloss_data.shape[1], patch_cols):
                

                sample = extract_sample(
                            forestloss_data,
                            forestloss_profile,
                            label_data,
                            i,
                            j,
                            patch_rows,
                            patch_cols,
                            target_years,
                            target_values,
                            min_ratio_for_driver,
                            forest_loss_window,
                            min_px_forest_loss)
                
                
                if sample: #If conditions are not met, it returns None; if not, it contains the three items
                    driver_class, forestloss, patch_geometry = sample
                    drivers.append(int(driver_class))
                    
                    write_numpy_array_to_tif(
                        array=forestloss, 
                        geometry=patch_geometry, 
                        profile=forestloss_profile,
                        patch_size=(patch_rows, patch_cols),
                        output_path=os.path.join(forestloss_patches_dir, f'{sampleid}.tif')
                    )
                    geoms.append(patch_geometry)
                    sampleids.append(sampleid)
                    sampleid += 1

        if len(label_gdf) == 1: #For e.g., MapBiomas, i want prints per TMF tile 
            print(f"Items: {len(geoms)}")
    
    print(f"Items: {len(geoms)}")

                

0
28
Items: 47897
29
Items: 48181
Items: 48181


In [64]:
gdf = gpd.GeoDataFrame(geometry=geoms)

In [65]:
gdf['driver'] = [class_mapping[val] for val in drivers]
gdf['sampleid'] = sampleids

In [66]:
gdf

,geometry,driver,sampleid
0,"POLYGON ((-6.81639 8.55672, -6.81639 8.56615, ...",cocoa,3000000
1,"POLYGON ((-6.88241 8.51899, -6.88241 8.52842, ...",cocoa,3000001
2,"POLYGON ((-6.88241 8.50956, -6.88241 8.51899, ...",cocoa,3000002
3,"POLYGON ((-6.78809 8.47183, -6.78809 8.48126, ...",cocoa,3000003
4,"POLYGON ((-6.77866 8.47183, -6.77866 8.48126, ...",cocoa,3000004
...,...,...,...
48176,"POLYGON ((0.52834 6.87777, 0.52834 6.88720, 0....",cocoa,3048176
48177,"POLYGON ((0.54721 6.87777, 0.54721 6.88720, 0....",cocoa,3048177
48178,"POLYGON ((0.63210 6.74572, 0.63210 6.75515, 0....",cocoa,3048178
48179,"POLYGON ((0.50948 6.73629, 0.50948 6.74572, 0....",cocoa,3048179


In [123]:
forest_loss_years_list = [ (item_tuple[0].tolist(),item_tuple[1].tolist()) for item_tuple in forest_loss_years]
gdf['forest_loss_years'] = forest_loss_years_list


In [67]:
df = pd.DataFrame(gdf)

df.to_csv(f'../data/tmp/{dataset}.csv', index=False)


In [68]:
gdf.to_file(f'../data/tmp/{dataset}.gpkg')

In [69]:
dataset

'cocoa'

In [94]:
#gdf[['geometry', 'main_driver']].to_file('../../data/tmp/cocoa_2000_2020.gpkg')

/tmp/ipykernel_99284/3068935073.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf[['geometry', 'main_driver']].to_file('../../data/tmp/annotation_vancutsem_2000_2020.shp')


In [51]:
def convert_string_to_lists(input_string):
    try:
        lists = ast.literal_eval(input_string)
        if isinstance(lists, tuple) and len(lists) == 2 and all(isinstance(lst, list) for lst in lists):
            return lists
        else:
            return None
    except (SyntaxError, ValueError):
        return None

In [52]:
import re
import ast
def modify_string_multiple_items(input_string):
    # Adjusting regex to match and replace patterns for arrays with multiple items
    modified_string = re.sub(r'array\(\[([0-9, ]+)\](, dtype=\w+\d*)?\)', r'[\1]', input_string)
    # Removing any trailing unmatched parentheses
    modified_string = re.sub(r'\)+$', '', modified_string)
    modified_string = modified_string.replace('(','')
    return ast.literal_eval(modified_string)
    #return [eval(item.strip()) for item in  modified_string.split(',')]

# Testing with the new input
output_str_multiple = modify_string_multiple_items('(array([2017, 2020, 2021], dtype=uint16), array([2, 3, 5]))')
output_str_multiple

([2017, 2020, 2021], [2, 3, 5])

In [53]:
gdf['forest_loss_years'] = gdf['forest_loss_years'].apply(convert_string_to_lists)


### Extract a small subset of the data to get started



In [84]:
df = pd.read_csv(f'../data/labels/{dataset}.csv')

In [85]:
    
def flatten(nested_list):
    return [x for xs in nested_list for x in xs]

if dataset == 'mapbiomas':
    driver_classes = np.unique(df.driver, return_counts=True)[0]
    max_count_per_class = 1000


    indices = []
    for _class in driver_classes:
        try:
            class_samples = df.loc[df.driver == _class].sample(max_count_per_class, random_state=1).index
        except ValueError:
            class_samples = df.loc[df.driver == _class].index
        indices.append(class_samples.tolist())

    indices = flatten(indices)

    subset = df.loc[indices]
else:
    subset = df.sample(1000, random_state=1)

In [90]:
pd.DataFrame(subset).to_csv(f'../data/labels/dataset_small/{dataset}_1000.csv', index=False)
#subset[['geometry', 'main_driver_name']].to_file('../../data/starting_dataset/cocoa_1000.gpkg')

In [ ]:
if data